In [40]:
!pip install deep_sort_realtime
!pip install "paddleocr>=2.0.

ERROR: Invalid requirement: 'paddleocr>=2.0.': Expected end or semicolon (after version specifier)
    paddleocr>=2.0.
             ~~~~~^


In [41]:
import fileinput
import os
from pathlib import Path
from typing import Union
import torch
import cv2 as cv
import numpy as np
import re
import matplotlib.pyplot as plt
from deep_sort_realtime.deepsort_tracker import DeepSort

In [42]:
# 1. Download YOLO11 weights if not present
if not os.path.isfile('yolo11s.pt'):
    weights_url = 'https://github.com/ultralytics/assets/releases/download/v8.2.0/yolo11s.pt'
    os.system(f'wget {weights_url}')

# 2. Download example images if not present
if not os.path.isdir('examples'):
    examples_url = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip'
    os.system(f'wget {examples_url} -O coco128.zip')
    os.system('unzip coco128.zip -d examples')
    os.system('rm -rf coco128.zip')

# 3. Clone YOLO11 repo if not present
if not os.path.isdir('yolo11'):
    yolov11_repo_url = 'https://github.com/ultralytics/ultralytics.git'
    os.system(f'git clone {yolov11_repo_url} yolo11')

In [43]:
# from ultralytics import YOLO
# from pathlib import Path
# import torch
# import os

# # Settings
# yolo11s_weights = "yolo11s.pt"            # default YOLO11s weights
# custom_weights = r"E:\Datasets\ultralytics-main\runs\train_yolo11s\weights\best.pt"  # your custom weights
# device_id = 0
# image_size = 640
# image_folder = r"E:\image\CarLicense\test_images"
# save_folder = r"E:\image\CarLicense\results"
# os.makedirs(save_folder, exist_ok=True)

# # Load YOLO11s pretrained model
# device = f'cuda:{device_id}' if torch.cuda.is_available() else 'cpu'
# print(f'Using device: {device}')
# model = YOLO(yolo11s_weights)  # load pretrained YOLO11s
# model.to(device)

# # Load custom weights on top (non-strict to keep unmatched layers from YOLO11s)
# model.model.load_state_dict(torch.load(custom_weights, map_location=device)["model"], strict=False)

# # Warm-up (optional)
# dummy_input = torch.zeros(1, 3, image_size, image_size).to(device)
# _ = model.model(dummy_input)
# model.model.eval()

# # Process images in the folder
# image_paths = list(Path(image_folder).glob("*.*"))

# for img_path in image_paths:
#     results = model(str(img_path), imgsz=image_size)
#     r = results[0]                   # get the actual result
#     r.show()                          # display image with bounding boxes
#     r.save(Path(save_folder) / img_path.name)  # save image with original filename


In [44]:
# from ultralytics import YOLO
# import torch

# # Paths
# base_weights = "yolo11s.pt"                  # Official YOLO11s weights
# custom_weights = r"E:\image\CarLicense\best.pt"  # Your fine-tuned weights

# # Device
# device = "cuda:0" if torch.cuda.is_available() else "cpu"
# print(f"Using device: {device}")

# # Load base model
# model = YOLO(base_weights).to(device)

# # Load your custom weights on top (non-strict, so unmatched layers are ignored)
# state_dict = torch.load(custom_weights, map_location=device)
# if "model" in state_dict:  # handle Ultralytics format
#     state_dict = state_dict["model"]

# model.model.load_state_dict(state_dict, strict=False)

# # Run inference
# results = model(r"E:\image\CarLicense\test_images")
# for r in results:
#     r.show()
#     r.save(r"E:\image\CarLicense\results_merge")


In [48]:
import torch
from ultralytics import YOLO
from pathlib import Path
from torchvision.ops import nms
import os

# Paths
base_weights = "yolo11s.pt"                       # pre-trained YOLO11s
custom_weights = r"E:\Datasets\ultralytics-main\runs\train_yolo11s\weights\best.pt"   # your fine-tuned model
image_folder = r"E:\image\CarLicense\test_images"
save_folder = r"E:\image\CarLicense\merged_results"
os.makedirs(save_folder, exist_ok=True)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load models
model_base = YOLO(base_weights).to(device)
model_custom = YOLO(custom_weights).to(device)

# Get class names
coco_classes = model_base.names
custom_classes = model_custom.names

# Run inference
results_base = model_base(image_folder, save=False, verbose=False)
results_custom = model_custom(image_folder, save=False, verbose=False)

# Combine predictions with NMS
for idx, (rb, rc) in enumerate(zip(results_base, results_custom)):
    preds = torch.cat((rb.boxes.data, rc.boxes.data), dim=0)  # (x1,y1,x2,y2,conf,cls)

    if preds.numel() == 0:
        print(f"No detections in image {rb.path}")
        continue

    # Apply TorchVision NMS
    boxes = preds[:, :4]
    scores = preds[:, 4]
    keep = nms(boxes, scores, iou_threshold=0.5)
    final_preds = preds[keep]

    # Replace boxes in one result object (e.g., rb)
    rb.boxes.data = final_preds

    # Save merged annotated image
    save_path = Path(save_folder) / Path(rb.path).name
    rb.save(filename=save_path)

    print(f"Processed and saved: {save_path}")


Using device: cuda
Processed and saved: E:\image\CarLicense\merged_results\Cars0.png
Processed and saved: E:\image\CarLicense\merged_results\Cars1.png
Processed and saved: E:\image\CarLicense\merged_results\Cars2.png
Processed and saved: E:\image\CarLicense\merged_results\Cars47.png
Processed and saved: E:\image\CarLicense\merged_results\Cars48.png
Processed and saved: E:\image\CarLicense\merged_results\Cars51.png
Processed and saved: E:\image\CarLicense\merged_results\Cars52.png
Processed and saved: E:\image\CarLicense\merged_results\Cars55.png
Processed and saved: E:\image\CarLicense\merged_results\Cars9.png
Processed and saved: E:\image\CarLicense\merged_results\Example-of-Labels-Car-make-model-year-color-and-other-information_Q320.jpg
Processed and saved: E:\image\CarLicense\merged_results\e16530094c899a7c4ff1287f02fd.jpg
Processed and saved: E:\image\CarLicense\merged_results\images.jpg
Processed and saved: E:\image\CarLicense\merged_results\istockphoto-1833955010-612x612.jpg
Proc

In [54]:
import cv2 as cv
import torch
from ultralytics import YOLO
from ultralytics.utils.ops import scale_coords
from torchvision.ops import nms
import numpy as np
import os

# -------------------------
# Helper: Letterbox function
# -------------------------
def letterbox(img, new_size=640, stride=32, color=(114,114,114)):
    h, w = img.shape[:2]
    r = new_size / max(h, w)
    new_unpad = int(round(w * r)), int(round(h * r))
    dw, dh = new_size - new_unpad[0], new_size - new_unpad[1]
    dw /= 2
    dh /= 2

    img_resized = cv.resize(img, new_unpad, interpolation=cv.INTER_LINEAR)
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    img_padded = cv.copyMakeBorder(img_resized, top, bottom, left, right, cv.BORDER_CONSTANT, value=color)
    return img_padded, r, (dw, dh)

# -------------------------
# Paths
# -------------------------
image_path = r"E:\image\CarLicense\merged_results\Cars0.png"
output_path = r"E:\image\CarLicense\results\Cars0_annotated.png"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# -------------------------
# Device
# -------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# -------------------------
# Load Models
# -------------------------
model_base = YOLO("yolo11s.pt").to(device)
model_custom = YOLO(r"E:\Datasets\ultralytics-main\runs\train_yolo11s\weights\best.pt").to(device)

# Class names
coco_classes = model_base.names
custom_classes = model_custom.names

# -------------------------
# Load Image
# -------------------------
source_image = cv.imread(image_path)
if source_image is None:
    raise FileNotFoundError(f"Image not found: {image_path}")
print("Original shape:", source_image.shape)

# Letterbox resize
img_size = 640
img, ratio, pad = letterbox(source_image, new_size=img_size)
img_tensor = torch.from_numpy(img).permute(2,0,1).unsqueeze(0).float().to(device) / 255.0

# -------------------------
# Inference
# -------------------------
results_base = model_base(img_tensor)
results_custom = model_custom(img_tensor)

# -------------------------
# Merge Predictions + NMS
# -------------------------
preds = torch.cat((results_base[0].boxes.data, results_custom[0].boxes.data), dim=0)
final_preds = preds
if preds.numel() > 0:
    boxes = preds[:, :4]
    scores = preds[:, 4]
    keep = nms(boxes, scores, iou_threshold=0.5)
    final_preds = preds[keep]

# Scale back to original image size
if final_preds.numel() > 0:
    final_preds[:, :4] = scale_coords(img_tensor.shape[2:], final_preds[:, :4], source_image.shape).round()

# -------------------------
# Draw and label boxes
# -------------------------
for i, det in enumerate(final_preds):
    x1, y1, x2, y2, conf, cls_id = det.tolist()
    # Determine label
    if i < len(results_base[0].boxes.data):
        label = coco_classes[int(cls_id)]
    else:
        label = custom_classes[int(cls_id)]
    
    cv.rectangle(source_image, (int(x1), int(y1)), (int(x2), int(y2)), (0,255,0), 2)
    cv.putText(source_image, f"{label} {conf:.2f}", (int(x1), int(y1)-5),
               cv.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

# -------------------------
# Save Annotated Image
# -------------------------
cv.imwrite(output_path, source_image)
print(f"Saved annotated image to {output_path}")


Using device: cuda
Original shape: (268, 500, 3)

0: 640x640 1 car, 57.0ms
Speed: 0.1ms preprocess, 57.0ms inference, 4.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 licence, 14.2ms
Speed: 0.0ms preprocess, 14.2ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 640)
Saved annotated image to E:\image\CarLicense\results\Cars0_annotated.png
